In [13]:
import torch
import torch.nn as nn
import torch.optim as optim

# ==========================================
# 1. ПОДГОТОВКА ДАННЫХ (Реальные цитаты)
# ==========================================
# Класс 0: Лев Толстой (светло, масштабно, природа, общество)
tolstoy_texts = [
    "князь андрей подошел к старому дубу",
    "пьер безухов долго смотрел на темное небо",
    "наташа ростова радостно танцевала на балу",
    "граф илья андреевич тепло улыбался гостям",
    "огромная французская армия наступала на москву",
    "светский вечер у анны павловны начался",
    "солнце ярко освещало широкое поле аустерлица"
]

# Класс 1: Федор Достоевский (мрачно, психология, напряжение, теснота)
dostoevsky_texts = [
    "раскольников крепко сжал топор под пальто",
    "соня мармеладова тихо плакала в темном углу",
    "старуха процентщица подозрительно приоткрыла дверь",
    "порфирий петрович хитро прищурился и промолчал",
    "свидригайлов мрачно смотрел в грязное окно",
    "он проснулся в своей тесной желтой каморке",
    "дрожащие руки студента покрылись холодным потом"
]

all_texts = tolstoy_texts + dostoevsky_texts
labels = [0]*len(tolstoy_texts) + [1]*len(dostoevsky_texts)

# Простейший токенизатор (собираем словарь из слов)
vocab = {"<PAD>": 0}
for text in all_texts:
    for word in text.split():
        if word not in vocab:
            vocab[word] = len(vocab)

# Переводим текст в числа и выравниваем длину предложений до 7 слов
max_len = 7
X_data = []
for text in all_texts:
    tokens = [vocab.get(w, 0) for w in text.split()]
    # Паддинг (добиваем нулями, если короче 7)
    tokens = tokens + [0] * (max_len - len(tokens))
    X_data.append(tokens[:max_len])

X_tensor = torch.tensor(X_data, dtype=torch.long)
y_tensor = torch.tensor(labels, dtype=torch.long)

In [14]:
# ==========================================
# 2. АРХИТЕКТУРА МОДЕЛИ НА ATTENTION
# ==========================================
class AuthorAttentionClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_heads, num_classes):
        super().__init__()
        # TODO 1: Инициализируй слой nn.Embedding с padding_idx=0
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        
        # TODO 2: Инициализируй слой nn.MultiheadAttention.
        # Подсказка: передай embed_dim и num_heads. 
        # ОБЯЗАТЕЛЬНО укажи аргумент batch_first=True, иначе PyTorch запутается в размерностях!
        self.attention = nn.MultiheadAttention(embed_dim=embed_dim, num_heads=num_heads, batch_first=True)

        # TODO 3: Инициализируй финальный классификатор nn.Linear
        # Он должен принимать embed_dim и выдавать num_classes
        self.fc = nn.Linear(embed_dim, num_classes)
        pass

    def forward(self, x):
        # x имеет форму [batch_size, seq_len]
        
        # TODO 4: Получи эмбеддинги слов (пропусти x через embedding)
        emb = self.embedding(x)
        
        # TODO 5: Пропусти эмбеддинги через слой внимания.
        out, weights = self.attention(query=emb, key=emb, value=emb)
        
        # TODO 6: Сжатие последовательности (Pooling).
        pooled_out = out.mean(dim=1)
        
        # TODO 7: Пропусти сжатый вектор через nn.Linear и верни логиты
        logits = self.fc(pooled_out)
        return logits

In [15]:
# ==========================================
# 3. ОБУЧЕНИЕ МОДЕЛИ (Training Loop)
# ==========================================
# Гиперпараметры
EMBED_DIM = 16
NUM_HEADS = 2  # 16 отлично делится на 2
NUM_CLASSES = 2
EPOCHS = 30

model = AuthorAttentionClassifier(len(vocab), EMBED_DIM, NUM_HEADS, NUM_CLASSES)

# TODO 8: Создай функцию потерь (используй nn.CrossEntropyLoss)
criterion = nn.CrossEntropyLoss()

# TODO 9: Создай оптимизатор Adam, передав ему параметры модели (model.parameters()) и lr=0.01
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)

print("Начинаем обучение...")
for epoch in range(EPOCHS):
    # TODO 10: Сделай forward pass (передай X_tensor в модель)
    out = model(X_tensor)

    # TODO 11: Вычисли ошибку (передай логиты и y_tensor в функцию потерь)
    loss = criterion(out, y_tensor)

    # TODO 12: Очисти старые градиенты с помощью метода .zero_grad() у оптимизатора
    optimizer.zero_grad()

    # TODO 13: Сделай backward pass (вызови .backward() у объекта ошибки)
    loss.backward()

    # TODO 14: Сделай шаг оптимизатора (вызови .step())
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()
    
    if (epoch + 1) % 5 == 0:
        print(f"Эпоха {epoch+1}/{EPOCHS}, Loss: {loss.item():.4f}")

# Проверим, как модель предсказывает на обучающей выборке
with torch.no_grad():
    final_logits = model(X_tensor)
    predictions = torch.argmax(final_logits, dim=1)
    acc = (predictions == y_tensor).float().mean().item()
    print(f"\nФинальная точность на текстах: {acc * 100:.1f}%")

Начинаем обучение...
Эпоха 5/30, Loss: 0.3884
Эпоха 10/30, Loss: 0.0425
Эпоха 15/30, Loss: 0.0002
Эпоха 20/30, Loss: 0.0000
Эпоха 25/30, Loss: 0.0000
Эпоха 30/30, Loss: 0.0000

Финальная точность на текстах: 100.0%
